In [ ]:
!pip install Bio

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.4/276.4 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 43.1 MB/s eta 0:00:00


In [ ]:
!pip install Levenshtein

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.1/174.1 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 32.7 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Running this notebook is computationally intensive. You'll need to be running the GPU docker. Additionally, ensure you have plenty of disk space, and ideally, multiple CPUs available. Intermediate results have been stored and are accessible on S3. See the Supp_Fig_2.ipynb notebook for accessing and plotting this data.

In [ ]:
import sys
sys.path

['/content',
 '/env/python',
 '/usr/lib/python310.zip',
 '/usr/lib/python3.10',
 '/usr/lib/python3.10/lib-dynload',
 '',
 '/usr/local/lib/python3.10/dist-packages',
 '/usr/lib/python3/dist-packages',
 '/usr/local/lib/python3.10/dist-packages/IPython/extensions',
 '/root/.ipython']

In [ ]:
import os
import sys
import warnings
import multiprocessing as mp
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy


# #sys.path.append('../common')
py_file_location1 = "/content/drive/MyDrive/Colab Notebooks/Marjan/Low N/analysis/A003_policy_optimization/"
sys.path.append(os.path.abspath(py_file_location1))

py_file_location2 = "/content/drive/MyDrive/Colab Notebooks/Marjan/Low N/analysis/common/"
sys.path.append(os.path.abspath(py_file_location2))

import data_io_utils
import paths
import utils
import constants

import A003_common
import policy_evaluation
import acquisition_policies
import models

%reload_ext autoreload
%autoreload 2

ModuleNotFoundError: ignored

In [ ]:
paths.DATASETS_DIR

'/content/drive/MyDrive/Colab Notebooks/Marjan/Low N/analysis/common/../../data/s3/datasets'

In [ ]:
#data_io_utils.sync_s3_path_to_local(paths.DATASETS_DIR)

In [ ]:
#data_io_utils.sync_s3_path_to_local(paths.POLICY_EVAL_DIR)

In [ ]:
#data_io_utils.sync_s3_path_to_local(paths.EVOTUNING_CKPT_DIR)

## Config

In [ ]:
random.seed(8329)
np.random.seed(4158)

FORCE = True

N_REPLICATES = 20
SPLIT = 2 # Split 0 used for training, 1 for prospective design, 2 for final figure.

training_sets = ['sarkisyan']
acq_policies = ['random']
n_training_points_schedule = np.array([8, 24, 96])

models = [

    'EvotunedUniRep_Random_Init_1_LassoLars',
    'EvotunedUniRep_Random_Init_1_Ridge',
    'EvotunedUniRep_Random_Init_1_RidgeSparseRefit',
    'EvotunedUniRep_Random_Init_1_EnsembledRidgeSparseRefit'
]

#models = [
#     'LassoLars',
#     'Ridge',
#     'RidgeSparseRefit',
#     'EnsembledRidgeSparseRefit',

#     'Doc2VecLassoLars',
#     'Doc2VecRidge',
#     'Doc2VecRidgeSparseRefit',
#     'Doc2VecEnsembledRidgeSparseRefit',

#     'UniRepLassoLars',
#     'UniRepRidge',
#     'UniRepRidgeSparseRefit',
#     'UniRepEnsembledRidgeSparseRefit',

    # 'EvotunedUniRep_Random_Init_1_LassoLars',
    # 'EvotunedUniRep_Random_Init_1_Ridge',
    # 'EvotunedUniRep_Random_Init_1_RidgeSparseRefit',
    # 'EvotunedUniRep_Random_Init_1_EnsembledRidgeSparseRefit'

#     'EvotunedUniRep_Global_Init_1_LassoLars',
#     'EvotunedUniRep_Global_Init_1_Ridge',
#     'EvotunedUniRep_Global_Init_1_RidgeSparseRefit',
#     'EvotunedUniRep_Global_Init_1_EnsembledRidgeSparseRefit',

#     'EvotunedUniRep_Global_Init_2_LassoLars',
#     'EvotunedUniRep_Global_Init_2_Ridge',
#     'EvotunedUniRep_Global_Init_2_RidgeSparseRefit',
#     'EvotunedUniRep_Global_Init_2_EnsembledRidgeSparseRefit',
# ]

## Run

In [ ]:
for model in models:
    print("00000000000000000000000000000000000000000000000000")
    for acq_policy in acq_policies:
        print("1111111111111111111111111111111111111111111111111111")
        for training_set in training_sets:
            print("2222222222222222222222222222222222222222222222222")

            ## Load inputs
            inputs = policy_evaluation.load_data_eff_inputs(
                split=SPLIT,
                training_set_name=training_set,
                acq_policy=acq_policy,
                model=model)

            print('CHANGED N_TRAINING_POINTS_SCHEDULE')
            inputs['n_training_points_schedule'] = n_training_points_schedule

            ## Sync any previous progress from S3
            print(inputs['root_output_dir'])
            if data_io_utils.path_exists_on_s3(inputs['root_output_dir']):
                print('Found previous data on S3.')
                data_io_utils.sync_s3_path_to_local(inputs['root_output_dir'])
            else:
                print('No previous data found on S3.')

            ## RUN
            for i in range(N_REPLICATES):
                results = policy_evaluation.evaluate_model_and_acquisition_policy(
                    inputs['training_set_df'],
                    inputs['acquisition_policy_obj'],
                    inputs['n_training_points_schedule'],
                    inputs['acquisition_policy_params'],
                    inputs['model_obj'],
                    inputs['generalization_set_dfs'],
                    inputs['generalization_set_names'],
                    inputs['generalization_set_sub_category_columns'],
                    inputs['generalization_set_calc_params'],
                    os.path.join(inputs['root_output_dir'], 'rep_' + str(i)), # subdir for replicate
                    force=FORCE,
                    verbose=True,
                    save_models=True
                )

00000000000000000000000000000000000000000000000000
1111111111111111111111111111111111111111111111111111
2222222222222222222222222222222222222222222222222
Split: 2
training_set_name: sarkisyan
aqc_policy:random
model: EvotunedUniRep_Random_Init_1_LassoLars
/content/drive/MyDrive/Colab Notebooks/Marjan/Low N/analysis/common/../../data/s3/datasets/for_acquisition/all_seqs_sark_synneigh_fphomologs_ET_RANDOM_INIT_1_avg_hidden.npy


FileNotFoundError: ignored

## Sync to S3

In [ ]:
data_io_utils.sync_local_path_to_s3(paths.POLICY_EVAL_DIR)